# Step 1: Load and Aggregate NCDB 2019

**Input:** NCDB 2019 (one row per person involved in a collision)
**Output:** `ncdb2019_collisions.csv` (one row per collision, with features and target `Casualty_Class`)

Code meanings come from the NCDB Data Dictionary:
https://open.canada.ca/data/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a/resource/21eb7966-38da-4814-a80e-521bce6c4c27/download/data_dictionary.pdf

## 1. Load the raw data

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = ("https://open.canada.ca/data/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a/"
            "resource/1e2a6aa0-c35f-4736-984a-71f2eac1c1dc/download/2019_dataset_en.csv")

# Load everything as text so codes like "01" and "UU" stay intact
raw = pd.read_csv(DATA_URL, dtype=str)
raw.columns = raw.columns.str.strip().str.upper()

print("Shape:", raw.shape)
raw.head()

Shape: (272301, 23)


,C_YEAR,C_MNTH,C_WDAY,C_HOUR,C_SEV,C_VEHS,C_CONF,C_RCFG,C_WTHR,C_RSUR,...,V_TYPE,V_YEAR,P_ID,P_SEX,P_AGE,P_PSN,P_ISEV,P_SAFE,P_USER,C_CASE
0,2019,1,1,11,2,2,32,1,1,5,...,1,2015,1,M,56,QQ,1,2,U,2681651
1,2019,1,1,11,2,2,32,1,1,5,...,1,2015,UU,U,UU,UU,U,UU,U,2681651
2,2019,1,1,11,2,2,32,1,1,5,...,1,2010,1,F,30,11,2,2,1,2681651
3,2019,1,1,11,2,2,32,1,1,5,...,1,2010,2,U,1,QQ,N,NN,U,2681651
4,2019,1,1,16,2,2,32,1,2,5,...,1,2007,1,F,27,11,2,2,1,2681652


In [ ]:
EXPECTED = ["C_CASE", "C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_SEV", "C_VEHS",
            "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF",
            "V_ID", "V_TYPE", "V_YEAR", "P_ID", "P_SEX", "P_AGE", "P_PSN",
            "P_ISEV", "P_SAFE", "P_USER"]

missing_cols = [c for c in EXPECTED if c not in raw.columns]
extra_cols = [c for c in raw.columns if c not in EXPECTED]
print("Missing expected columns:", missing_cols or "none")
print("Extra columns:", extra_cols or "none")

Missing expected columns: none
Extra columns: none


## 2. Convert NCDB missing codes

NCDB uses letter codes instead of blanks:
- `U` / `UU` / `UUUU` = unknown, `X` / `XX` = not provided by jurisdiction -> treated as **missing (NaN)**
- `Q` / `QQ` = "other" -> kept as its own category
- `N` / `NN` = not applicable -> kept as its own category

In [ ]:
df = raw.copy()
for col in df.columns:
    df[col] = df[col].str.strip()

# Values made only of U and/or X characters are missing
missing_mask = df.apply(lambda s: s.str.fullmatch(r"[UX]+", na=False))

print("Missing codes per column (person level):")
summary = pd.DataFrame({
    "n_missing": missing_mask.sum(),
    "pct_missing": (missing_mask.mean() * 100).round(2),
}).sort_values("pct_missing", ascending=False)
display(summary)

df = df.mask(missing_mask)

Missing codes per column (person level):


,n_missing,pct_missing
P_SAFE,30686,11.27
P_AGE,17444,6.41
P_SEX,15792,5.80
C_RCFG,14818,5.44
P_USER,14769,5.42
C_RALN,13761,5.05
C_TRAF,13706,5.03
V_YEAR,11148,4.09
C_CONF,8378,3.08
P_PSN,6322,2.32


## 3. Person-level flags

- `P_ISEV`: 1 = no injury, 2 = injury, 3 = fatality, N = not applicable (e.g. parked car placeholder)
- `P_USER`: 4 = bicyclist; `V_TYPE`: 17 = bicycle
- `V_TYPE` heavy vehicles: 07 unit trucks > 4536 kg, 08 road tractors, 09-11 buses
- `P_SAFE`: 01 = no safety device used, 13 = no device (not available)

In [ ]:
HEAVY_TYPES = {"07", "08", "09", "10", "11"}   # edit if you define "heavy" differently
NO_SAFETY = {"01", "13"}
SAFETY_KNOWN = {"01", "02", "09", "10", "11", "12", "13"}

df["is_injury"] = df["P_ISEV"].eq("2")
df["is_fatal"] = df["P_ISEV"].eq("3")
df["isev_unknown"] = df["P_ISEV"].isna()
df["is_cyclist"] = df["P_USER"].eq("4") | df["V_TYPE"].eq("17")
df["is_heavy"] = df["V_TYPE"].isin(HEAVY_TYPES)
df["no_safety"] = df["P_SAFE"].isin(NO_SAFETY)
df["safety_known"] = df["P_SAFE"].isin(SAFETY_KNOWN)

df[["is_injury", "is_fatal", "isev_unknown", "is_cyclist", "is_heavy", "no_safety"]].mean().round(4)

,0
is_injury,0.5171
is_fatal,0.0065
isev_unknown,0.0217
is_cyclist,0.0172
is_heavy,0.0103
no_safety,0.0112


## 4. Aggregate to one row per collision

In [ ]:
COLLISION_COLS = ["C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_SEV", "C_VEHS",
                  "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF"]

# Sanity check: collision-level fields should be the same for every person in a collision
n_unique = df.groupby("C_CASE")[COLLISION_COLS].nunique()
print("Collisions with inconsistent collision-level values:")
print((n_unique > 1).sum())

Collisions with inconsistent collision-level values:
C_YEAR    0
C_MNTH    0
C_WDAY    0
C_HOUR    0
C_SEV     0
C_VEHS    0
C_CONF    0
C_RCFG    0
C_WTHR    0
C_RSUR    0
C_RALN    0
C_TRAF    0
dtype: int64


In [ ]:
agg_spec = {c: (c, "first") for c in COLLISION_COLS}
agg_spec.update(
    Num_Persons=("P_ID", "size"),
    Total_Injuries=("is_injury", "sum"),
    Total_Fatalities=("is_fatal", "sum"),
    Unknown_Severity_Persons=("isev_unknown", "sum"),
    Heavy_Vehicle_Present=("is_heavy", "max"),
    Cyclist_Present=("is_cyclist", "max"),
    Any_No_Safety_Device=("no_safety", "max"),
    Safety_Known_Persons=("safety_known", "sum"),
)

col = df.groupby("C_CASE").agg(**agg_spec).reset_index()

# Safety device flag: 1 if anyone had no device, 0 if all known people used one,
# NaN if nobody's safety device status is known
col["Safety_Device_Not_Used"] = np.where(
    col["Any_No_Safety_Device"], 1,
    np.where(col["Safety_Known_Persons"] > 0, 0, np.nan))
col = col.drop(columns=["Any_No_Safety_Device", "Safety_Known_Persons"])

for c in ["Heavy_Vehicle_Present", "Cyclist_Present"]:
    col[c] = col[c].astype(int)

print("Collisions:", len(col))
col.head()

Collisions: 105782


,C_CASE,C_YEAR,C_MNTH,C_WDAY,C_HOUR,C_SEV,C_VEHS,C_CONF,C_RCFG,C_WTHR,C_RSUR,C_RALN,C_TRAF,Num_Persons,Total_Injuries,Total_Fatalities,Unknown_Severity_Persons,Heavy_Vehicle_Present,Cyclist_Present,Safety_Device_Not_Used
0,2681637,2019,1,7,14,2,2,QQ,1,1,1,1,18,5,1,0,2,0,0,NaN
1,2681638,2019,1,2,9,2,2,2,1,1,1,1,18,3,1,0,1,0,0,NaN
2,2681639,2019,1,6,10,2,2,32,1,1,1,1,None,2,1,0,0,0,0,NaN
3,2681640,2019,1,3,8,2,2,31,3,1,3,1,None,2,1,0,0,0,0,NaN
4,2681641,2019,1,4,13,2,2,3,1,4,5,1,18,2,1,0,0,0,0,NaN


## 5. Derived outcome variables and type conversion

In [ ]:
col["Total_Casualties"] = col["Total_Injuries"] + col["Total_Fatalities"]
col["Fatal_Collision"] = (col["Total_Fatalities"] > 0).astype(int)

col["Casualty_Class"] = pd.cut(col["Total_Casualties"], bins=[0, 1, 2, np.inf],
                               labels=["1", "2", "3+"], right=True)
# Collisions with 0 known casualties get NaN (should be rare: NCDB only has injury/fatal collisions)

for c in ["C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_VEHS"]:
    col[c] = pd.to_numeric(col[c], errors="coerce")

# Intersection flag (C_RCFG 02 = public road intersection, 03 = driveway/parking lot entrance,
# 09 = traffic circle). Kept as a flag so you can decide later whether to filter.
col["Is_Intersection"] = col["C_RCFG"].isin(["02", "03", "09"]).astype(int)

col.dtypes

,0
C_CASE,object
C_YEAR,int64
C_MNTH,float64
C_WDAY,float64
C_HOUR,float64
C_SEV,object
C_VEHS,int64
C_CONF,object
C_RCFG,object
C_WTHR,object


## 6. Sanity checks

In [ ]:
print("Casualty_Class distribution (%):")
print((col["Casualty_Class"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nZero-casualty collisions:", (col["Total_Casualties"] == 0).sum())
print("Collisions with >=1 person of unknown severity:", (col["Unknown_Severity_Persons"] > 0).sum())

# C_SEV: 1 = fatal collision, 2 = non-fatal injury collision
check = pd.crosstab(col["C_SEV"], col["Fatal_Collision"])
print("\nC_SEV vs our Fatal_Collision (should line up):")
print(check)

print("\nIntersection collisions:", col["Is_Intersection"].sum(),
      f"({col['Is_Intersection'].mean() * 100:.1f}%)")

Casualty_Class distribution (%):
Casualty_Class
1      76.05
2      17.10
3+      6.85
NaN     0.00
Name: proportion, dtype: float64

Zero-casualty collisions: 1
Collisions with >=1 person of unknown severity: 3323

C_SEV vs our Fatal_Collision (should line up):
Fatal_Collision       0     1
C_SEV                        
1                     0  1623
2                104159     0

Intersection collisions: 0 (0.0%)


In [ ]:
print("Missing values per column (collision level, %):")
(col.isna().mean() * 100).round(2).sort_values(ascending=False)

Missing values per column (collision level, %):


,0
Safety_Device_Not_Used,96.23
C_RCFG,5.71
C_TRAF,5.49
C_RALN,5.07
C_CONF,3.26
C_WTHR,1.42
C_RSUR,0.97
C_HOUR,0.81
C_VEHS,0.00
C_SEV,0.00


## 7. Save

In [ ]:
OUT = "ncdb2019_collisions.csv"
col.to_csv(OUT, index=False)
print("Saved", OUT, col.shape)

# In Colab, this downloads the file to your computer
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass

Saved ncdb2019_collisions.csv (105782, 24)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>